In [2]:
from transformers import GPT2LMHeadModel,GPT2Tokenizer
from datasets import load_dataset
import torch
import torch.nn as nn

In [3]:
device='cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
tok=GPT2Tokenizer.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
tok.pad_token=tok.eos_token

In [6]:
mod=GPT2LMHeadModel.from_pretrained('gpt2').to(device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
mod.transformer.h

ModuleList(
  (0-11): 12 x GPT2Block(
    (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (attn): GPT2Attention(
      (c_attn): Conv1D(nf=2304, nx=768)
      (c_proj): Conv1D(nf=768, nx=768)
      (attn_dropout): Dropout(p=0.1, inplace=False)
      (resid_dropout): Dropout(p=0.1, inplace=False)
    )
    (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (mlp): GPT2MLP(
      (c_fc): Conv1D(nf=3072, nx=768)
      (c_proj): Conv1D(nf=768, nx=3072)
      (act): NewGELUActivation()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)

In [8]:
class pLe(nn.Module):
    def __init__(self,voc,dm,dp):
        super().__init__()
        self.toke=nn.Embedding(voc,dp)
        self.cont=nn.Linear(dm,dp)
        self.outt=nn.Linear(dp,dm)
    def forward(self,tokid,h0):
        et=self.toke(tokid)
        econ=self.cont(h0)
        return self.outt(et+econ)

In [9]:
voc=mod.config.vocab_size
dm=mod.config.n_embd
nl=len(mod.transformer.h)
pllay=nn.ModuleList([pLe(voc,dm,dp=64) for _ in range(nl)])

In [10]:
org=mod.transformer.h

In [32]:
def forw(inp,attnm=None,label=None):
    b,t=inp.shape
    pos=torch.arange(t,device=device).unsqueeze(0)
    h0=mod.transformer.wte(inp)+mod.transformer.wpe(pos)
    hidd=h0
    for i,block in enumerate(mod.transformer.h):
        hidd=block(hidd)[0]
        ple=pllay[i](inp,h0)
        hidd=hidd+ple
    hidd=mod.transformer.ln_f(hidd)
    logits=mod.lm_head(hidd)
    loss=None
    if label is not None:
        sftl=logits[:,:-1,:].contiguous()
        stfla=label[:,1:].contiguous()
        loss=nn.functional.cross_entropy(sftl.view(-1,sftl.size(-1)),stfla.view(-1))
    return logits,loss

In [33]:
optim=torch.optim.Adam([*mod.parameters(),*pllay.parameters()],lr=1e-5)

In [13]:
da=load_dataset('roneneldan/TinyStories')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [14]:
def token(d):
    return tok(d['text'],truncation=True,padding='max_length',max_length=128)

In [15]:
da=da.map(token,batched=True,remove_columns=['text'])

Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [17]:
da.set_format(type='torch')

In [19]:
train_loader = torch.utils.data.DataLoader(da["train"],batch_size=8,shuffle=True)

In [23]:
pllay=pllay.to(device)

In [34]:
mod.train()

for epoch in range(1):
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits, loss = forw(
            input_ids,
            attnm=attention_mask,
            label=input_ids
        )

        optim.zero_grad()
        loss.backward()
        optim.step()

        if step % 500 == 0:
            print(f"Step {step}, Loss: {loss.item():.4f}")

Step 0, Loss: 4.2824
Step 50, Loss: 2.7736
Step 100, Loss: 2.5296
Step 150, Loss: 2.3467
Step 200, Loss: 2.4917
Step 250, Loss: 2.2077
Step 300, Loss: 2.2329
Step 350, Loss: 2.2033
Step 400, Loss: 1.9067
Step 450, Loss: 2.3613
Step 500, Loss: 2.3671
Step 550, Loss: 2.0876
Step 600, Loss: 2.0513
Step 650, Loss: 2.0356
Step 700, Loss: 2.1971
Step 750, Loss: 2.0843
Step 800, Loss: 2.2957
Step 850, Loss: 2.1316
Step 900, Loss: 1.9957
Step 950, Loss: 2.1104
Step 1000, Loss: 1.9155
Step 1050, Loss: 2.0608
Step 1100, Loss: 2.0201
Step 1150, Loss: 2.0977
Step 1200, Loss: 2.1514
Step 1250, Loss: 2.1189
Step 1300, Loss: 1.9333
Step 1350, Loss: 1.8343
Step 1400, Loss: 1.9617
Step 1450, Loss: 2.0026
Step 1500, Loss: 1.8787
Step 1550, Loss: 2.1025
Step 1600, Loss: 2.0085
Step 1650, Loss: 2.3037
Step 1700, Loss: 1.8233
Step 1750, Loss: 1.7287
Step 1800, Loss: 2.0344
Step 1850, Loss: 2.0106
Step 1900, Loss: 1.9209
Step 1950, Loss: 2.1734
Step 2000, Loss: 2.1039
Step 2050, Loss: 1.8828
Step 2100, Loss

KeyboardInterrupt: 

In [35]:
torch.save({
    "model": mod.state_dict(),
    "ple": pllay.state_dict()
}, "gpt2_ple_tinystories.pt")

In [36]:
mod.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [41]:
prompt = "She and her husband were in the shower together"
input_ids = tok(prompt, return_tensors="pt").input_ids.to(device)

for _ in range(500):
    logits, _ = forw(input_ids)
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    input_ids = torch.cat([input_ids, next_token], dim=1)

print(tok.decode(input_ids[0]))

She and her husband were in the shower together. They were having so much fun. They were having so much fun.

Suddenly, a big, scary monster appeared. It was a big, scary monster. It was big and scary. It had a big, scary face.

"What is that?" asked Tom.

"I don't know," said Tom. "Maybe it is a monster. Maybe it is a big, scary monster."

Tom was scared. He wanted to go home. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He ran to the bathroom and opened the door. He saw a big, scary monster. He was scared. He 

In [42]:
prompt = "Once upon a time, there was a little dog named Max."

# Bad - greedy decoding
greedy = mod.generate(**tok(prompt, return_tensors="pt").to(device), max_new_tokens=50)

# Good - sampling with repetition penalty
good = mod.generate(
    **tok(prompt, return_tensors="pt").to(device),
    max_new_tokens=500,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
    repetition_penalty=1.3,
    no_repeat_ngram_size=2
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [44]:
tok.decode(good[0])

"Once upon a time, there was a little dog named Max. He lived in the backyard with his family and loved to play outside every day. One sunny morning, he saw an old tree growing taller than him! \nMax wanted to climb on it so badly that they couldn't get out of bed before sunrise. So instead we just sat under its branches for hours until suddenly something magical happened: when our mom came into the room from her porch looking like she had taken shelter inside someone's house - but without even touching them? I thought maybe this might be what could happen if you were lucky enough not to touch my home first thingâ€™s ashes...<|endoftext|>"